> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


## Production imports

Once `mrta.evaluation` and `mrta.observability.logging` are implemented:

```python
from mrta.evaluation.eval_pipeline import run_eval
from mrta.evaluation.metrics import answer_relevance, faithfulness, citation_correctness, hallucination_rate
from mrta.observability.logging import StructuredLogger
```

See [`../../production-ready.md`](../../production-ready.md) — Phase 09.


# Phase 9 — Evaluation, Observability & Docker

**Goal:** Make the project look senior-level. We add three production-shaped layers on top of the working RAG system:

1. **Evaluation** — a small benchmark + Ragas metrics for groundedness, faithfulness, and answer relevance.
2. **Observability** — structured JSONL logs with per-run latency, tokens, cited pages, and retrieval traces.
3. **Deployment** — a `docker-compose.yml` that brings up backend + frontend + Qdrant with one command.

Most candidates skip these. That is exactly why doing them is portfolio gold.


## 9.1 — Build a tiny benchmark

Five hand-labeled QA pairs over *Attention Is All You Need*. In a real project you would have 50-200; this is enough to show the pipeline.


In [ ]:
import os, sys, json
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

BENCH = Path("data/processed/benchmark.jsonl")
BENCH.parent.mkdir(parents=True, exist_ok=True)

benchmark = [
    {
        "question": "What is the dimensionality of the model and the feed-forward inner layer in the base Transformer?",
        "expected_pages": [3, 5],
        "expected_substrings": ["512", "2048"],
    },
    {
        "question": "How many attention heads does the base Transformer use?",
        "expected_pages": [5],
        "expected_substrings": ["8"],
    },
    {
        "question": "What positional encoding does the Transformer use and why?",
        "expected_pages": [6],
        "expected_substrings": ["sinusoid"],
    },
    {
        "question": "What BLEU score did the base Transformer achieve on WMT 2014 English-to-German?",
        "expected_pages": [8],
        "expected_substrings": ["27.3", "BLEU"],
    },
    {
        "question": "Why is scaled dot-product attention scaled by 1/sqrt(d_k)?",
        "expected_pages": [4],
        "expected_substrings": ["soft", "magnitude"],
    },
]

with BENCH.open("w") as f:
    for r in benchmark:
        f.write(json.dumps(r) + "\n")
print(f"Wrote {len(benchmark)} questions to {BENCH}")


## 9.2 — Custom metrics (no external deps)

Start with metrics you can compute yourself. They are interpretable and need no LLM judge.


In [ ]:
import re, faiss, time
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel
from typing import Optional
import ollama

class VectorStore:
    def __init__(self, dim, embedder):
        self.index = faiss.IndexFlatIP(dim); self.metadata=[]; self.embedder=embedder
    def search(self, q, k=5):
        v = self.embedder.encode([q], normalize_embeddings=True).astype("float32")
        s, i = self.index.search(v, k)
        return [{**self.metadata[idx], "score": float(s[0][r])} for r, idx in enumerate(i[0]) if idx >= 0]
    @classmethod
    def load(cls, p, embedder):
        p = Path(p); cfg = json.loads((p/"config.json").read_text())
        st = cls(cfg["dim"], embedder)
        st.index = faiss.read_index(str(p/"index.faiss"))
        st.metadata = [json.loads(l) for l in (p/"metadata.jsonl").read_text().splitlines()]
        return st

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
store = VectorStore.load("data/vector_store/aiayn", embedder)

SYSTEM = "You answer questions grounded in the provided context. Always cite pages."

def run_question(q: str, k: int = 5) -> dict:
    hits = store.search(q, k=k)
    context = "\n\n".join(f"[chunk {i+1} | page {h['page']}]\n{h['text']}" for i, h in enumerate(hits))
    prompt = f"--- CONTEXT ---\n{context}\n\n--- QUESTION ---\n{q}\n\n--- ANSWER (with [page X] citations) ---"
    t0 = time.time()
    resp = ollama.chat(model="llama3.2:3b",
                       messages=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],
                       options={"temperature":0.1})
    answer = resp["message"]["content"]
    cited = sorted({int(m.group(1)) for m in re.finditer(r"\[page (\d+)", answer)})
    return {
        "question": q, "answer": answer, "cited_pages": cited,
        "retrieved_pages": [h["page"] for h in hits],
        "retrieval_scores": [h["score"] for h in hits],
        "latency_s": time.time() - t0,
        "prompt_tokens": resp.get("prompt_eval_count"),
        "completion_tokens": resp.get("eval_count"),
    }

def metric_recall_at_k(expected_pages, retrieved_pages, k=5) -> float:
    if not expected_pages: return 0.0
    hits = sum(1 for p in expected_pages if p in retrieved_pages[:k])
    return hits / len(expected_pages)

def metric_substring_match(answer: str, expected: list[str]) -> float:
    if not expected: return 1.0
    return sum(1 for s in expected if s.lower() in answer.lower()) / len(expected)

def metric_citation_in_retrieval(cited, retrieved) -> float:
    if not cited: return 0.0
    return sum(1 for c in cited if c in retrieved) / len(cited)

print("Metrics defined.")


## 9.3 — Run the benchmark


In [ ]:
import pandas as pd

rows = []
for q in benchmark:
    res = run_question(q["question"])
    rows.append({
        "question": q["question"][:60] + "...",
        "recall@5": round(metric_recall_at_k(q["expected_pages"], res["retrieved_pages"]), 2),
        "substr_match": round(metric_substring_match(res["answer"], q["expected_substrings"]), 2),
        "cite_grounded": round(metric_citation_in_retrieval(res["cited_pages"], res["retrieved_pages"]), 2),
        "latency_s": round(res["latency_s"], 1),
        "cited": res["cited_pages"],
    })
df = pd.DataFrame(rows)
df


## 9.4 — Ragas for LLM-judged groundedness

[Ragas](https://github.com/explodinggradients/ragas) computes:

- **faithfulness** — every claim in the answer is supported by the context.
- **answer_relevancy** — the answer addresses the question.
- **context_precision** — retrieved chunks are useful.

It uses an LLM as judge. We point it at Ollama to keep everything local.

> Note: Ragas wraps several LangChain components. The exact setup snippet below is the simplest illustrative path; the version of Ragas you install may have a slightly different API surface, so consult its docs.


In [ ]:
# Illustrative — uncomment and adapt to your installed ragas version.
# from datasets import Dataset
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision
# from langchain_community.chat_models import ChatOllama
# from langchain_community.embeddings import HuggingFaceEmbeddings
#
# judge_llm = ChatOllama(model="llama3.2:3b", temperature=0)
# judge_emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
#
# rows = []
# for q in benchmark:
#     res = run_question(q["question"])
#     ctx = [store.metadata[i]["text"] for i in range(min(5, len(store.metadata)))]
#     rows.append({"question": q["question"], "answer": res["answer"], "contexts": ctx, "ground_truth": ""})
#
# ds = Dataset.from_list(rows)
# scores = evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision],
#                   llm=judge_llm, embeddings=judge_emb)
# print(scores)
print("See the cell above for the Ragas template; uncomment and adapt for your installed version.")


## 9.5 — Structured logging

Every production run goes through one logger. The format is JSONL: one event per line, machine-parseable, append-only.


In [ ]:
import structlog, json
from datetime import datetime, timezone

LOG_FILE = Path("data/logs/runs.jsonl")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

structlog.configure(
    processors=[
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso", utc=True),
        structlog.processors.JSONRenderer(),
    ],
    logger_factory=structlog.WriteLoggerFactory(file=LOG_FILE.open("a")),
)
log = structlog.get_logger()

def log_run(run: dict) -> None:
    log.info("rag_run",
             question=run["question"],
             latency_s=round(run["latency_s"], 2),
             cited_pages=run["cited_pages"],
             retrieved_pages=run["retrieved_pages"],
             prompt_tokens=run.get("prompt_tokens"),
             completion_tokens=run.get("completion_tokens"))

# Replay the benchmark and log.
for q in benchmark:
    log_run(run_question(q["question"]))

print("Last 3 log entries:")
for line in LOG_FILE.read_text().splitlines()[-3:]:
    print(" ", line[:200], "...")


## 9.6 — Aggregating logs

A 5-line analyzer that turns the JSONL into a dashboard-ready DataFrame:


In [ ]:
import pandas as pd

events = [json.loads(line) for line in LOG_FILE.read_text().splitlines()]
df_logs = pd.DataFrame(events)
print("Total runs logged:", len(df_logs))
df_logs[["timestamp", "question", "latency_s", "cited_pages"]].tail(5)


## 9.7 — Dockerfiles

Two Dockerfiles — one per process. Keeping them separate lets you scale the LLM-heavy API independently of the UI.


In [ ]:
DOCKER_DIR = Path("docker"); DOCKER_DIR.mkdir(exist_ok=True)

(DOCKER_DIR / "Dockerfile.api").write_text(r'''FROM python:3.11-slim

# System deps for PyMuPDF, tesseract (OCR), and pdfplumber.
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential libgl1 libglib2.0-0 tesseract-ocr poppler-utils \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY pyproject.toml .
COPY src/ ./src/
RUN pip install --no-cache-dir ".[api]"

COPY apps/api/ ./apps/api/
COPY .env.example .env

EXPOSE 8000
CMD ["uvicorn", "apps.api.main:app", "--host", "0.0.0.0", "--port", "8000"]
''', encoding="utf-8")

(DOCKER_DIR / "Dockerfile.ui").write_text(r'''FROM python:3.11-slim

WORKDIR /app
COPY pyproject.toml .
COPY src/ ./src/
RUN pip install --no-cache-dir ".[streamlit]"

COPY apps/streamlit/ ./apps/streamlit/

EXPOSE 8501
CMD ["streamlit", "run", "apps/streamlit/app.py", "--server.address=0.0.0.0"]
''', encoding="utf-8")

print("Wrote:", *DOCKER_DIR.glob("*"))


## 9.8 — docker-compose for the full stack

Three services:

- **api** — FastAPI app.
- **ui** — Streamlit frontend.
- **qdrant** — vector store for the production swap.

Ollama runs on the host (not in a container) because GPU pass-through is platform-specific and `ollama serve` on the host already speaks HTTP.


In [ ]:
(DOCKER_DIR / "docker-compose.yml").write_text(r'''services:
  api:
    build:
      context: ..
      dockerfile: docker/Dockerfile.api
    ports:
      - "8000:8000"
    environment:
      - OLLAMA_HOST=http://host.docker.internal:11434
      - VECTOR_STORE=qdrant
      - QDRANT_URL=http://qdrant:6333
    volumes:
      - ../data:/app/data
    depends_on:
      - qdrant

  ui:
    build:
      context: ..
      dockerfile: docker/Dockerfile.ui
    ports:
      - "8501:8501"
    environment:
      - API_URL=http://api:8000
    depends_on:
      - api

  qdrant:
    image: qdrant/qdrant:latest
    ports:
      - "6333:6333"
    volumes:
      - qdrant_storage:/qdrant/storage

volumes:
  qdrant_storage:
''', encoding="utf-8")
print("Wrote docker/docker-compose.yml")
print("\nBring up the stack:")
print("  cd docker && docker compose up --build")


## 9.9 — The Qdrant swap

Once the vector store is Qdrant, the `VectorStore` class only changes inside its methods — the public API stays the same. That is the payoff for designing an interface in Phase 3 rather than calling FAISS directly from the RAG pipeline.

Sketch:

```python
from qdrant_client import QdrantClient, models

class QdrantStore:
    def __init__(self, url: str, dim: int, embedder, collection: str = "papers"):
        self.client = QdrantClient(url=url)
        if collection not in [c.name for c in self.client.get_collections().collections]:
            self.client.create_collection(
                collection_name=collection,
                vectors_config=models.VectorParams(size=dim, distance=models.Distance.COSINE),
            )
        self.collection = collection
        self.embedder = embedder

    def add(self, chunks):
        embs = self.embedder.encode([c.text for c in chunks], normalize_embeddings=True)
        self.client.upsert(self.collection, points=[
            models.PointStruct(id=i, vector=v.tolist(), payload=c.model_dump())
            for i, (c, v) in enumerate(zip(chunks, embs))
        ])

    def search(self, query: str, k: int = 5, doc_id: str | None = None):
        q = self.embedder.encode([query], normalize_embeddings=True)[0].tolist()
        flt = models.Filter(must=[models.FieldCondition(key="doc_id", match=models.MatchValue(value=doc_id))]) if doc_id else None
        res = self.client.search(self.collection, query_vector=q, query_filter=flt, limit=k)
        return [{**r.payload, "score": r.score} for r in res]
```

Now the `VECTOR_STORE` env var swaps backends with no callsite changes.


## 9.10 — CI on GitHub Actions

Drop this file at `.github/workflows/ci.yml`:


In [ ]:
CI = Path(".github/workflows"); CI.mkdir(parents=True, exist_ok=True)
(CI / "ci.yml").write_text(r'''name: CI

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install --upgrade pip
      - run: pip install -r requirements.txt
      - run: pip install ruff black pytest
      - run: ruff check app tests
      - run: black --check app tests
      - run: pytest -q
''', encoding="utf-8")
print("Wrote .github/workflows/ci.yml")


## 9.11 — README polish checklist

A senior-quality README hits all of these. Tick them off before announcing the repo:

- [ ] One-sentence elevator pitch at the top.
- [ ] A screenshot or animated GIF of the app working.
- [ ] An architecture diagram (Excalidraw or draw.io). Save as `docs/architecture.png`.
- [ ] A "Why these choices?" section — FAISS vs Qdrant, local vs API, chunk size.
- [ ] A reproducible **Quick start** that gets a stranger to a running demo in <5 minutes.
- [ ] An evaluation section with at least one table of metrics.
- [ ] A "Limitations" section that names what you would do next.
- [ ] A demo video link (5–8 min walkthrough on YouTube/Loom).

Honesty about limitations is the strongest senior signal — it shows you know where the project would break.


## What you learned

- A custom-metrics evaluation harness with `recall@k`, substring grounding, and citation-in-retrieval rate.
- How Ragas plugs an LLM judge into the pipeline (and that you should use a local Ollama judge to stay free).
- Structured JSONL logging with `structlog` for replayable runs.
- A two-Dockerfile + `docker-compose.yml` setup with Qdrant as the production vector store.
- The Qdrant swap sketch — payoff for designing a clean `VectorStore` interface earlier.
- A GitHub Actions CI workflow.
- A README polish checklist.

## Exercises

1. Expand the benchmark to 30 questions across 3 papers (e.g. AIAYN, BERT, GPT-2 / GPT-3).
2. Add a `latency-p95` SLO to CI: fail the build if it regresses above 10s.
3. Pipe the JSONL logs into a small Streamlit dashboard (`docs/dashboard.py`) that shows recall, latency, and cost over time.
4. Record a 5-minute demo video and link it from the README.

---

That's the full tutorial series. You now have:

1. A repo scaffold with `src/mrta/`, `apps/`, `docker/`, CI, and typed config.
2. Ten runnable tutorial notebooks (`notebooks/2026-05-25-phase00 ... phase09`).
3. An end-to-end working multimodal RAG assistant on local models.
4. The eval, observability, and deployment scaffolding to justify the "production-ready" label on your README.

Good luck — go ship it.
